# ML Project

## Data Cleaning `teams.csv`

In [ ]:
import pandas as pd

teams = pd.read_csv('../dataset/teams.csv')

# Detect and remove unused columns
nan_columns = teams.columns[teams.isna().all()]

print("\nColumns containing only NaN values:")
print(nan_columns.tolist())

teams = teams.drop(columns=nan_columns)

# Detect and remove columns with all zeros
zero_columns = teams.columns[(teams == 0).all()]
print("\nColumns containing only zeros:")
print(zero_columns.tolist())

teams = teams.drop(columns=zero_columns)

# Keeping statistical outliers is important because:
# - High values represent exceptionally good performing teams
# - Low values represent underperforming teams

teams.to_csv('../dataset_cleaned/teams.csv', index=False)


## Data Cleaning `teams_post.csv`

In [ ]:
teams_post = pd.read_csv('../dataset/teams_post.csv')

# Remove unused columns like 'lgID'
teams_post = teams_post.drop(columns=['lgID'])

teams_post.to_csv('../dataset_cleaned/teams_post.csv', index=False)


## Data Cleaning `awards_players.csv`

In [ ]:
awards_players = pd.read_csv('../dataset/awards_players.csv')

# Remove unused columns like 'lgID'
awards_players = awards_players.drop(columns=['lgID'])

awards_players.to_csv('../dataset_cleaned/awards_players.csv', index=False)


## Data Cleaning `series_post.csv`

In [ ]:
series_post = pd.read_csv('../dataset/series_post.csv')

# Remove unused columns like 'lgIDWinner' and 'lgIDLoser'
series_post = series_post.drop(columns=['lgIDWinner', 'lgIDLoser'])

series_post.to_csv('../dataset_cleaned/series_post.csv', index=False)


## Data Cleaning `players.csv`

In [ ]:
players = pd.read_csv('../dataset/players.csv')

# Remove columns that only have 'zero' values 
zero_columns = players.columns[(players == 0).all()] 
print("\nColumns containing only zeros:")
print(zero_columns.tolist())

players = players.drop(columns=zero_columns)

# Remove players that are not assigned to any team

players_teams = pd.read_csv('../dataset/players_teams.csv')['playerID'].unique()
players = players[players['bioID'].isin(players_teams)]

players.to_csv('../dataset_cleaned/players.csv', index=False)

# !!! Some players don't have weight (need to calculate using prediction models or other methods) !!!


## Data Cleaning `players_teams.csv`

In [ ]:
players_teams = pd.read_csv('../dataset/players_teams.csv')

# Remove 'lgID' column as it's not needed
players_teams = players_teams.drop(columns=['lgID'])

# Remove players that haven't played a single game or minute
players_teams = players_teams[(players_teams['GP'] > 0) | (players_teams['minutes'] > 0)]

players_teams.to_csv('../dataset_cleaned/players_teams.csv', index=False)

# !!! Understand what 'PF' and 'DQ' means !!!


## Data Cleaning `coaches.csv`

In [ ]:
import pandas as pd

coaches = pd.read_csv('../dataset/coaches.csv')

# Remove 'lgID' column as it's not needed
coaches = coaches.drop(columns=['lgID'])

# Remove the coaches that had 0 wins and 0 losses (never coached a game)
coaches = coaches[(coaches['won'] > 0) | (coaches['lost'] > 0)]

# Assuming 'coaches' is your current DataFrame

# 1. Identify the maximum stint number for every team-year combination
# This gives us a DataFrame containing the max 'stint' for each group
max_stints = coaches.groupby(['tmID', 'year'])['stint'].max().reset_index()
max_stints = max_stints.rename(columns={'stint': 'max_stint'})

# 2. Merge the max stint back into the original DataFrame
df_merged = pd.merge(
    coaches, 
    max_stints, 
    on=['tmID', 'year'], 
    how='left'
)

# 3. Filter the DataFrame to keep only the rows where the current 'stint' equals the 'max_stint'
# This keeps the final coach's record for multi-coach years.
df_final_coaches = df_merged[df_merged['stint'] == df_merged['max_stint']].copy()

# 4. Drop the helper column and ensure there are no duplicates left
df_final_coaches = df_final_coaches.drop(columns=['max_stint'])

#Drop existing one that has the same max_stint
df_final_coaches = df_final_coaches.drop_duplicates(
    subset=['tmID', 'year'], 
    keep='first'
)

# Optional check: Verify that (tmID, year) pairs are now unique
if df_final_coaches.duplicated(['tmID', 'year']).any():
    print("Warning: Duplicate team-year entries still exist after filtering.")
else:
    print(f"Successfully reduced the dataset from {len(coaches)} to {len(df_final_coaches)} unique team-year records.")

# Update your coaches variable to use the cleaned data
coaches = df_final_coaches

coaches.to_csv('../dataset_cleaned/coaches.csv', index=False)
